# Task 6, File 1: Build Dataset and Shared Config

Builds `non_trend_english.csv` (posts outside the 5 known trends only) and saves the shared
preprocessing config (`lda_config.json`) that `01a_lda_text.ipynb` and `01b_lda_hashtags.ipynb`
both load, so the stopword/exclusion lists have one source of truth.

Full design reasoning: `TASK6_LDA_design_doc.md`. Exact steps: `TASK6_LDA_STEP_BY_STEP.md`.

## Step 0: Design confirmed with project owner

- Data source: `non_trend_english.csv`, posts outside the 5 known trends only -- NOT
  `trends_combined_english.csv`, NOT the full English-filtered dataset.
- Text and hashtags LDA run as separate files (`01a`, `01b`), concurrently.
- Full dataset, no sampling.
- K tested empirically (5, 10, 15, 20, 25) per file, not guessed.
- `ngram_range=(1,1)` (unigrams only) -- revised from an earlier `(1,2)` draft; the 8
  originally-planned trend-phrase bigram exclusions were confirmed to have zero occurrences in
  `non_trend_english.csv` (it's already everything outside the 5 trends by construction), so the
  bigram-exclusion mechanism did no real work while adding a matrix/vocabulary mismatch risk and
  a coherence-scoring complication. See `TASK6_LDA_design_doc.md` for the full reasoning.
- Lemmatization via NLTK's `WordNetLemmatizer`.

In [1]:
import pandas as pd
import json

## Step 1: Build `non_trend_english.csv`

In [2]:
df1 = pd.read_csv("../../data/manya_first_12012024.csv")
df2 = pd.read_csv("../../data/manya_0711-1104.csv")
df3 = pd.read_csv("../../data/manya_1105_1201.csv")
df4 = pd.read_csv("../../data/manya_0410-0710.csv")
df_full = pd.concat([df1, df2, df3, df4], ignore_index=True)
print("Step 1a - Raw combined:", df_full.shape)  # expect (298635, 19)

Step 1a - Raw combined: (298635, 19)


In [3]:
df_full = df_full[df_full["lang"] == "en"].copy()
print("Step 1b - After English filter:", df_full.shape)  # expect (207038, 19)

Step 1b - After English filter: (207038, 19)


In [4]:
trend_keywords = {
    "feta_pasta": ["feta pasta", "baked feta pasta", "baked feta"],
    "sourdough": ["sourdough", "sourdough starter", "sourdough bread"],
    "banana_bread": ["banana bread", "banana bread recipe", "bananabread"],
    "baked_oats": ["baked oats", "baked oatmeal", "bakedoats"],
    "dalgona_coffee": ["dalgona coffee", "whipped coffee", "dalgona"],
}
trend_ids = set()
for trend, kws in trend_keywords.items():
    pattern = "|".join(kws)
    mask = (df_full["text"].str.contains(pattern, case=False, na=False, regex=True) |
            df_full["hashtags"].str.contains(pattern, case=False, na=False, regex=True))
    trend_ids.update(df_full.loc[mask, "id"])

df_full = df_full[~df_full["id"].isin(trend_ids)].copy()
print("Step 1c - After excluding the 5 trend-matched posts:", df_full.shape)  # expect (81409, 19)

Step 1c - After excluding the 5 trend-matched posts: (81409, 19)


In [5]:
before = len(df_full)
df_full = df_full.dropna(subset=["text", "hashtags"], how="all").copy()
print("Step 1d - After dropping rows with neither text nor hashtags:", df_full.shape)
# expect (73933, 19), ~7,476 dropped
print(f"Dropped {before - len(df_full)} rows")

Step 1d - After dropping rows with neither text nor hashtags: (73933, 19)
Dropped 7476 rows


In [ ]:
df_full.to_csv("../../output/cleaned_data/non_trend_english.csv", index=False)
print("Saved non_trend_english.csv")

## Step 2: Save the shared preprocessing config as JSON

In [7]:
config = {
    "tokenization_fragments": ["s", "t", "m", "ve", "g"],
    "platform_artifacts": ["mention", "redacted", "link", "bio"],
    "recipe_boilerplate": ["recipe", "make", "made", "food", "like", "just", "delicious",
                            "love", "good", "day", "today", "time"],
    "units_measurements": ["cup", "tsp", "tbsp", "add", "salt", "oil", "minutes", "ingredients"],
    "trend_single_words": ["sourdough", "dalgona"],
    "min_df": 5,
    "max_df": 0.5,
    "ngram_range": [1, 1],
    "k_values_to_test": [5, 10, 15, 20, 25],
}

with open("lda_config.json", "w") as f:
    json.dump(config, f, indent=2)
print("Saved lda_config.json")
print(json.dumps(config, indent=2))

Saved lda_config.json
{
  "tokenization_fragments": [
    "s",
    "t",
    "m",
    "ve",
    "g"
  ],
  "platform_artifacts": [
    "mention",
    "redacted",
    "link",
    "bio"
  ],
  "recipe_boilerplate": [
    "recipe",
    "make",
    "made",
    "food",
    "like",
    "just",
    "delicious",
    "love",
    "good",
    "day",
    "today",
    "time"
  ],
  "units_measurements": [
    "cup",
    "tsp",
    "tbsp",
    "add",
    "salt",
    "oil",
    "minutes",
    "ingredients"
  ],
  "trend_single_words": [
    "sourdough",
    "dalgona"
  ],
  "min_df": 5,
  "max_df": 0.5,
  "ngram_range": [
    1,
    1
  ],
  "k_values_to_test": [
    5,
    10,
    15,
    20,
    25
  ]
}


Confirm these words are intentionally NOT in the config's exclusion lists (they should remain
usable in the vocabulary): `cheese`, `bread`, `coffee`, `banana`, `oats`, `oatmeal`, `chocolate`,
`baked`, `baking`, `feta`.

In [8]:
all_excluded = set()
for key in ["tokenization_fragments", "platform_artifacts", "recipe_boilerplate",
            "units_measurements", "trend_single_words"]:
    all_excluded.update(config[key])

must_keep = ["cheese", "bread", "coffee", "banana", "oats", "oatmeal", "chocolate",
             "baked", "baking", "feta"]
wrongly_excluded = [w for w in must_keep if w in all_excluded]
print("Words that must stay in the vocabulary, wrongly excluded (should be empty):", wrongly_excluded)
assert not wrongly_excluded

Words that must stay in the vocabulary, wrongly excluded (should be empty): []
